# Actualizar catalog.json de Orbit

Descarga TLEs frescos desde **CelesTrak** (desde tu máquina o cualquier red que no bloquee celestrak.org) y genera un `catalog.json` compatible con el formato que espera la app.

**Pasos:**
1. Ejecuta la celda de configuración.
2. Descarga los grupos que quieras.
3. Guarda el `catalog.json` resultante y súbelo/reemplaza el del proyecto.


In [ ]:
# ─── Instalación de dependencias ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])
print("✓ Dependencias listas")


## 1. Configuración — elige grupos y ruta de salida


In [ ]:
import os

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN  ← modifica esto según lo que quieras descargar
# ──────────────────────────────────────────────────────────────────────────────

# Ruta donde se guardará el catalog.json
CATALOG_OUTPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), "Orbit", "config", "catalog.json")
# Si ejecutas el notebook desde la raíz del proyecto, usa simplemente:
# CATALOG_OUTPUT_PATH = "config/catalog.json"

# Grupos de CelesTrak a descargar (FORMAT=tle)
# Lista completa disponible en: https://celestrak.org/NORAD/elements/
CELESTRAK_GROUPS = [
    "active",       # Todos los satélites activos (~6000)
    "stations",     # ISS, Tiangong, etc.
    "starlink",
    "oneweb",
    "geo",
    "gnss",
    "visual",
    "weather",
    "planet",
    "cubesat",
    "science",
    "military",
    "galileo",
    "goes",
    "noaa",
]

# Timeout por descarga en segundos
TIMEOUT_SECONDS = 30

# Número de descargas en paralelo
MAX_WORKERS = 6

print(f"✓ Configuración lista — {len(CELESTRAK_GROUPS)} grupos, salida: {CATALOG_OUTPUT_PATH}")


## 2. Descarga TLEs desde CelesTrak (paralelo)


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

CELESTRAK_BASE = "https://celestrak.org/NORAD/elements/gp.php"

def download_group(group: str) -> tuple[str, list[str], str | None]:
    """Descarga un grupo de CelesTrak y devuelve (grupo, líneas, error)."""
    url = f"{CELESTRAK_BASE}?GROUP={group}&FORMAT=tle"
    try:
        r = requests.get(url, timeout=TIMEOUT_SECONDS, headers={
            "User-Agent": "Orbit-Catalog-Updater/1.0",
            "Accept": "text/plain",
        })
        r.raise_for_status()
        lines = [l.strip() for l in r.text.splitlines() if l.strip()]
        return group, lines, None
    except Exception as e:
        return group, [], str(e)


# Descargar en paralelo
raw_lines: dict[str, list[str]] = {}
errors: dict[str, str] = {}

print(f"Descargando {len(CELESTRAK_GROUPS)} grupos con {MAX_WORKERS} workers en paralelo...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(download_group, g): g for g in CELESTRAK_GROUPS}
    for future in as_completed(futures):
        group, lines, err = future.result()
        if err:
            errors[group] = err
            print(f"  ✗ {group}: {err}")
        else:
            raw_lines[group] = lines
            print(f"  ✓ {group}: {len(lines)} líneas")

print(f"\nÉxito: {len(raw_lines)}/{len(CELESTRAK_GROUPS)} grupos | Fallos: {len(errors)}")


## 3. Parsear TLEs y construir entradas del catálogo


In [ ]:
import math

EARTH_RADIUS_KM = 6378.137
EARTH_MU_KM3_S2 = 398600.4418


def tle_checksum(line: str) -> int:
    total = 0
    for ch in line[:-1]:
        if ch.isdigit():
            total += int(ch)
        elif ch == "-":
            total += 1
    return total % 10


def is_valid_tle(line1: str, line2: str) -> bool:
    if not (line1.startswith("1 ") and line2.startswith("2 ")):
        return False
    if len(line1) < 69 or len(line2) < 69:
        return False
    return tle_checksum(line1) == int(line1[-1]) and tle_checksum(line2) == int(line2[-1])


def estimate_perigee_km(line2: str) -> float | None:
    """Estima el perigeo en km a partir del movimiento medio (line2)."""
    try:
        mean_motion_rev_day = float(line2[52:63])
        n_rad_s = mean_motion_rev_day * 2 * math.pi / 86400
        a_km = (EARTH_MU_KM3_S2 / (n_rad_s ** 2)) ** (1/3)
        ecc_str = line2[26:33].strip()
        ecc = float("0." + ecc_str)
        perigee = a_km * (1 - ecc) - EARTH_RADIUS_KM
        return round(perigee, 4)
    except Exception:
        return None


def parse_tle_lines(lines: list[str]) -> list[dict]:
    """Parsea líneas TLE crudas y devuelve entradas en formato catalog.json."""
    entries = []
    i = 0
    while i < len(lines):
        # Formato: nombre, línea1, línea2
        # El nombre puede estar en la línea anterior si no empieza por 1/2
        if i + 2 < len(lines):
            name_candidate = lines[i]
            l1 = lines[i + 1]
            l2 = lines[i + 2]
            if l1.startswith("1 ") and l2.startswith("2 ") and is_valid_tle(l1, l2):
                name = name_candidate if not name_candidate.startswith(("1 ", "2 ")) else f"NORAD-{l1[2:7].strip()}"
                entries.append({
                    "name": name.strip(),
                    "line1": l1,
                    "line2": l2,
                    "sourceFormat": "TLE",
                    "sourceOrigin": "CATALOG",
                    "operator": "unknown",
                    "owner": "unknown",
                    "perigee_km": estimate_perigee_km(l2),
                })
                i += 3
                continue
        i += 1
    return entries


# Parsear todas las líneas descargadas
all_entries: dict[str, dict] = {}  # clave = nombre, para deduplicar

for group, lines in raw_lines.items():
    parsed = parse_tle_lines(lines)
    for entry in parsed:
        all_entries[entry["name"]] = entry  # última aparición gana (más fresca)

entries_list = list(all_entries.values())
print(f"Total entradas únicas: {len(entries_list)}")
print(f"Ejemplo: {entries_list[0] if entries_list else 'N/A'}")
